# Making Use of Vectorization

这份 notebook 覆盖三部分：

1. Python 里的“非向量化”写法
2. NumPy 的向量化
3. pandas 的向量化，以及时间序列上的常见操作



## 1. Python 里没有真正的向量化

Python 的 list 不能像数学里的向量那样直接做标量乘法；`2 * v` 只是把列表复制一遍。

In [1]:
v = [1, 2, 3, 4, 5]

sm = [2 * i for i in v]
print("list comprehension:", sm)

print("2 * list:", 2 * v)


list comprehension: [2, 4, 6, 8, 10]
2 * list: [1, 2, 3, 4, 5, 1, 2, 3, 4, 5]


## 2. NumPy 提供真正的向量化

`ndarray` 可以直接做标量乘法、线性变换、幂运算，以及聚合运算。

In [2]:
import numpy as np

a = np.array(v)
print(a)
print(type(a))

print("2 * a =", 2 * a)
print("0.5 * a + 2 =", 0.5 * a + 2)


[1 2 3 4 5]
<class 'numpy.ndarray'>
2 * a = [ 2  4  6  8 10]
0.5 * a + 2 = [2.5 3.  3.5 4.  4.5]


In [3]:
a2 = np.arange(12).reshape((4, 3))
print(a2)

print("2 * a2 =\n", 2 * a2)
print("a2 ** 2 =\n", a2 ** 2)


[[ 0  1  2]
 [ 3  4  5]
 [ 6  7  8]
 [ 9 10 11]]
2 * a2 =
 [[ 0  2  4]
 [ 6  8 10]
 [12 14 16]
 [18 20 22]]
a2 ** 2 =
 [[  0   1   4]
 [  9  16  25]
 [ 36  49  64]
 [ 81 100 121]]


In [4]:
print("a2.mean() =", a2.mean())
print("np.mean(a2) =", np.mean(a2))
print("a2.mean(axis=0) =", a2.mean(axis=0))
print("np.mean(a2, axis=1) =", np.mean(a2, axis=1))


a2.mean() = 5.5
np.mean(a2) = 5.5
a2.mean(axis=0) = [4.5 5.5 6.5]
np.mean(a2, axis=1) = [ 1.  4.  7. 10.]


## 3. pandas 也支持向量化

pandas 基于 NumPy，因此很多向量化操作可以直接作用在 `DataFrame` 上。

In [5]:
import pandas as pd

a = np.arange(15).reshape(5, 3)
columns = list("abc")
index = pd.date_range("2021-07-01", periods=5, freq="B")

df = pd.DataFrame(a, columns=columns, index=index)
print(df)


             a   b   c
2021-07-01   0   1   2
2021-07-02   3   4   5
2021-07-05   6   7   8
2021-07-06   9  10  11
2021-07-07  12  13  14


In [10]:
print("\n2 * df =\n", 2 * df)
print("\ndf.sum() =\n", df.sum())
print("\ndf.mean() =\n", df.mean())
print("\nnp.mean(df) =\n", np.mean(df))



2 * df =
              a   b   c
2021-07-01   0   2   4
2021-07-02   6   8  10
2021-07-05  12  14  16
2021-07-06  18  20  22
2021-07-07  24  26  28

df.sum() =
 a    30
b    35
c    40
dtype: int64

df.mean() =
 a    6.0
b    7.0
c    8.0
dtype: float64

np.mean(df) =
 7.0


In [11]:
print("df['a'] + df['c'] =\n", df["a"] + df["c"])
print("0.5 * df.a + 2 * df.b - df.c =\n", 0.5 * df.a + 2 * df.b - df.c)


df['a'] + df['c'] =
 2021-07-01     2
2021-07-02     8
2021-07-05    14
2021-07-06    20
2021-07-07    26
Freq: B, dtype: int64
0.5 * df.a + 2 * df.b - df.c =
 2021-07-01     0.0
2021-07-02     4.5
2021-07-05     9.0
2021-07-06    13.5
2021-07-07    18.0
Freq: B, dtype: float64


In [12]:
mask_a = df["a"] > 5
print("df['a'] > 5 =\n", mask_a)
print()
print("df[df['a'] > 5] =\n", df[mask_a])


df['a'] > 5 =
 2021-07-01    False
2021-07-02    False
2021-07-05     True
2021-07-06     True
2021-07-07     True
Freq: B, Name: a, dtype: bool

df[df['a'] > 5] =
              a   b   c
2021-07-05   6   7   8
2021-07-06   9  10  11
2021-07-07  12  13  14


In [13]:
print("df['c'] > df['b'] =\n", df["c"] > df["b"])
print()
print("0.15 * df.a + df.b > df.c =\n", 0.15 * df.a + df.b > df.c)


df['c'] > df['b'] =
 2021-07-01    True
2021-07-02    True
2021-07-05    True
2021-07-06    True
2021-07-07    True
Freq: B, dtype: bool

0.15 * df.a + df.b > df.c =
 2021-07-01    False
2021-07-02    False
2021-07-05    False
2021-07-06     True
2021-07-07     True
Freq: B, dtype: bool


## 4. 金融示例：用向量化生成模拟价格路径

原文提到 `generate_sample_data()` 使用几何布朗运动（GBM）生成样本金融数据。下面给出一个可运行版本。

In [ ]:
def generate_sample_data(rows, cols, freq="1min", seed=42):
    rows = int(rows)
    cols = int(cols)
    rng = np.random.default_rng(seed)

    index = pd.date_range("2021-01-01", periods=rows, freq=freq)
    dt = (index[1] - index[0]) / pd.Timedelta("365D")

    r = 0.05
    sigma = 0.5

    increments = (
        (r - 0.5 * sigma ** 2) * dt
        + sigma * np.sqrt(dt) * rng.standard_normal((rows, cols))
    )
    raw = np.exp(np.cumsum(increments, axis=0))
    raw = raw / raw[0] * 100

    columns = [f"No{i}" for i in range(cols)]
    return pd.DataFrame(raw, index=index, columns=columns)

sample = generate_sample_data(5, 3, freq="D")
print(sample)


## 5. 小结

- Python 原生 list 不适合做真正的数值向量运算
- NumPy 的 `ndarray` 提供了高效的向量化
- pandas 继承了这种能力，特别适合时间序列和金融数据
- 向量化通常更简洁，也更快，适合回测和金融算法实现

本章的核心，就是把“循环思维”换成“数组思维”。
